#### In this notebook, we get all successfully scraped records from the AIFF website and put them into a data frame.

In [9]:
from pathlib import Path
import pandas as pd
import json

In [3]:
#Getting success records

# read project folder (1 folder up from where we are right now)
data_collection = Path("../data_collection")

# read scrape log
log = pd.read_csv(data_collection / "scrape_log.csv")

#keeping only successful logs
log_success = log[log["status"] == "success"]

# getting file NAMES from the successful logs in a list [] called success_files
success_files = log_success["file_name"].tolist()

# we have the file names of success files. Now we want the actual files, not just the names. For that, we need to define the path for python.
json_folder = data_collection / "raw_match_centres" # note that raw_match_centre is where all your downloaded JSONs are from your scraper

# getting success files and saving them in a list called records
records = []

for file in success_files: #looping on each file in success_files 
    with open(json_folder / file, "r", encoding="utf-8") as f: #adding them here
        records.append(json.load(f))

# sanity check 
len(records)
# 21,027 success records.

21027

In [8]:
records[0].keys()

dict_keys(['fixture', 'teams_data', 'tournament', 'team1_players', 'team2_players', 'squad1_list', 'squad2_list', 'events', 'team1_score', 'team2_score', 'report', 'lineup1', 'lineup2', 'official1', 'official2', 'mvp', 'stats_available'])

In [10]:
print(type(records[0]['fixture'])) #dictionary
print(type(records[0]['events'])) #list of dictionaries
print(type(records[0]['tournament'])) #dictionary
print(type(records[0]['team1_players'])) #list of dictionaries
print(type(records[0]['team2_players'])) #list of dictionaries

<class 'dict'>
<class 'list'>
<class 'dict'>
<class 'list'>
<class 'list'>


In [4]:
#creating empty rows for player-match dataset
player_rows = []

#labelling the data we need for each JSON in records
for record in records: #loop through each JSON (biggest loop)
    
    # READABILITY
    
    # Give shorter names to the relevant parts of the JSON
    fixture = record["fixture"] #now fixture dictionary in each JSON is called fixture
    tournament = record["tournament"] #now events list is called events in each JSON
    events = record["events"] #now tournament dictionary is called tournament

    # Combine both teams' player lists
    players = (
        record["team1_players"]
        + record["team2_players"]
    )

    # SUBSTITUTION LOGIC
    
    # Loop through every player listed for this match/this JSON
    for player in players:

        # If any player information is empty, flag it!
        t1_empty = len(record["team1_players"]) == 0
        t2_empty = len(record["team2_players"]) == 0
        one_team_empty = t1_empty != t2_empty

        # Start by assuming the player did not come on as a substitute
        substituted = False

        # Check every event in this match
        for event in events:

            # A type 4 event is a substitution.
            # sub_aiff_player_id is the player who came on.
            if (
                event.get("type") == 4
                and event.get("sub_aiff_player_id") == player.get("aiff_id")
            ): #if type = 4 and id is there,
                substituted = True
                break #if you find a match of substitution for this player, break this loop, don't check more events, go to appearance loop!

        # Build one player-match row
        row = {
            "match_id": fixture.get("id"),
            "competition_id": fixture.get("tournament_id"),
            "competition_name": fixture.get("tournament_name"),
            "match_date": fixture.get("date"),
            "match_venue": fixture.get("venue"),
            "match_stadium": fixture.get("stadium"),
            "ft": fixture.get("ft"),
            "ft_text": fixture.get("ft_text"),

            "team_type": tournament.get("team_type"),

            "player_name": player.get("player_name"),
            "player_id": player.get("player_id"),
            "player_aiff_id": player.get("aiff_id"),
            "player_team_id": player.get("team_id"),
            "status": player.get("status"),
            "one_team_player_data_missing": one_team_empty,

            "substituted": substituted,
        }

        # Add this row to the list
        player_rows.append(row)

# Convert all rows into a dataframe
player_matches = pd.DataFrame(player_rows)

In [5]:
print(len(records))

21027


In [6]:
len(player_matches['match_id'].value_counts())

18205

In [7]:
print(len(records) - len(player_matches['match_id'].value_counts()))
# there is a difference on 2822 records. one possible hypothesis is that for some entries, player entries are absent. 

2822


In [10]:
player_matches.to_csv("scraped_raw_data.csv")